In [1]:
# 환경 변수 로드
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="./env/.env")

api_key = os.getenv('OPENAI_API_KEY')

In [9]:
# Chat 모델 및 프롬프트 설정
from langchain.chat_models import ChatOpenAI
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature= 0.1
)   

# Few-shot 예시 데이터
examples = [
    {
        "movie": "해리포터와 마법사의 돌",
        "answer": """
감독: 크리스 콜럼버스
주요 출연진: 다니엘 래드클리프, 루퍼트 그린트, 엠마 왓슨, 리처드 해리스, 매기 스미스
예산: 약 1억 2,500만 달러
흥행 수익: 약 10억 2,400만 달러
장르: 판타지, 가족, 모험
줄거리: 자신이 마법사라는 사실을 모른 채 이모네 집에서 구박받으며 살던 고아 소년 해리 포터가 11살 생일에 호그와트 마법학교 입학 초청장을 받으며 벌어지는 이야기. 마법 세계의 경이로움을 경험하며 부모님의 죽음에 얽힌 비밀과 어둠의 마법사 볼드모트에 맞서게 됩니다. 
        """,
    },
    {
        "movie": "주토피아",
        "answer": """
감독: 바이론 하워드, 리치 무어
주요 출연진: 지니퍼 굿윈, 제이슨 베이트먼, 이드리스 엘바, 제니 슬레이트
예산: 약 1억 5,000만 달러
흥행 수익: 약 10억 2,500만 달러
장르: 애니메이션, 모험, 코미디, 추리
줄거리: 누구나 무엇이든 될 수 있는 도시 주토피아에서 최초의 토끼 경찰관이 된 주디 홉스는 연쇄 실종 사건을 맡게 됩니다. 그녀는 뻔뻔한 사기꾼 여우 닉 와일드와 협력하여 도시 이면에 숨겨진 거대한 음모를 파헤치며 진정한 평등과 편견에 대한 메시지를 찾아갑니다.
        """,
    },
    {
        "movie": "조커",
        "answer": """
감독: 토드 필립스
주요 출연진: 호아킨 피닉스, 로버트 드 니로, 재지 비츠, 프란시스 콘로이
예산: 약 5,500만 달러
흥행 수익: 약 10억 7,400만 달러
장르: 범죄, 드라마, 스릴러
줄거리: 고담시의 광대 아서 플렉은 코미디언을 꿈꾸며 살아가지만, 가난과 정신 질환, 그리고 사회의 냉대 속에서 점차 무너져 내립니다. 반복되는 비극적인 상황과 폭력적인 현실에 부딪히던 그는 결국 내면의 광기를 폭발시키며 희대의 악당 조커로 변모해가는 과정을 그립니다.
        """,
    },
]

# 예시들을 담을 메시지 포맷
example_format = ChatPromptTemplate.from_messages(
    [
        ("human", "{movie}에 대한 정보를 줘" ),
        ("ai", "{answer}")
    ]
)

few_shot_template = FewShotChatMessagePromptTemplate(
    example_prompt = example_format, 
    examples = examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", """너는 영화 백과사전이야. 사용자가 요청한 영화의 정보를 예시 형식을 지켜서 제공해줘. 실제로 존재하지 않는 영화 이름이거나 정보가 전혀 없을 때만 "해당 영화에 대한 정보를 찾을 수 없습니다."라고 답해."""), # 시스템 역할 정의
    few_shot_template, # 위에서 만든 예시 데이터 삽입
    ("human", "{movie}에 대한 정보를 줘") # 실제 사용자 질문
])

chain = final_prompt | chat

response = chain.invoke({
    "movie": "알라딘"
})

print(response.content)

감독: 가이 리치
주요 출연진: 메나 마수드, 나오미 스콧, 윌 스미스, 마리아나 노보아, 나샤
예산: 약 1억 8,000만 달러
흥행 수익: 약 10억 3,500만 달러
장르: 모험, 가족, 판타지, 뮤지컬
줄거리: 도시 아그라바에서 살아가는 무리한 소년 알라딘은 마법 램프를 소유한 지니의 도움을 받아 공주 자스민의 마음을 얻으려고 노력합니다. 그러나 악당 자파르와의 대결, 그리고 마법사 자펫의 음모로부터 자스민과 아그라바를 지키기 위한 모험을 벌이게 됩니다.
